In [2]:
from pathlib import Path
import pandas as pd
import seaborn as sns


DATA_PATH = Path("/Users/nehaharish/github/Thesis-mlops/data/prepared/base_clean.parquet")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Parquet not found: {DATA_PATH}")

# read parquet (pandas will pick available engine: pyarrow or fastparquet)
df = pd.read_parquet(DATA_PATH)

# quick inspection
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.describe(include='all').T)


Shape: (9496, 31)


,provider,region_snd_base,region_rcv_base,proto,size_snd,size_rcv,tput_mean,tput_std,tput_min,tput_max,...,loss_pctl25,loss_median,loss_pctl75,loss_pctl95,start,az_snd,az_rcv,snd_continent,rcv_continent,duration
0,AWS,<NA>,<NA>,tcp,m3.medium,m3.medium,48.822629,9.006007,2.860082,62.91278,...,0.0,0.0,0.0,0.0,2015-04-07 19:23:31,<NA>,<NA>,<NA>,<NA>,300.0
1,AWS,<NA>,<NA>,tcp,m3.medium,m3.medium,53.173571,5.904978,2.859755,62.91407,...,0.0,0.0,0.0,0.0,2015-04-07 19:25:02,<NA>,<NA>,<NA>,<NA>,300.0
2,AWS,<NA>,<NA>,tcp,m3.medium,m3.medium,64.460791,3.988834,6.196100,73.39800,...,0.0,0.0,0.0,0.0,2015-04-07 19:25:14,<NA>,<NA>,<NA>,<NA>,300.0
3,AWS,<NA>,<NA>,tcp,m3.medium,m3.medium,64.107558,4.690045,6.196745,72.87507,...,0.0,0.0,0.0,0.0,2015-04-07 19:25:39,<NA>,<NA>,<NA>,<NA>,300.0
4,AWS,<NA>,<NA>,tcp,m3.medium,m3.medium,53.983393,4.475636,2.860136,62.91616,...,0.0,0.0,0.0,0.0,2015-04-07 19:25:58,<NA>,<NA>,<NA>,<NA>,300.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9496 entries, 0 to 9495
Data columns (total 31 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   provider         9496 non-null   category      
 1   region_snd_base  8656 non-null   string        
 2   region_rcv_base  8656 non-null   string        
 3   proto            9496 non-null   category      
 4   size_snd         9496 non-null   category      
 5   size_rcv         9496 non-null   category      
 6   tput_mean        9492 non-null   float64       
 7   tput_std         9492 non-null   float64       
 8   tput_min         9492 non-null   float64       
 9   tput_max         9492 non-null   float64       
 10  tput_pctl5       9492 non-null   float64       
 11  tput_pctl25      9492 non-null   float64       
 12  tput_median      9492 non-null   float64       
 13  tput_pctl75      9492 non-null   float64       
 14  tput_pctl95      9492 non-null   float64

None

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
provider,9496,2,AWS,8640,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region_snd_base,8656,4,us-east-1,2464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region_rcv_base,8656,4,us-east-1,2392,NaN,NaN,NaN,NaN,NaN,NaN,NaN
proto,9496,2,tcp,7624,NaN,NaN,NaN,NaN,NaN,NaN,NaN
size_snd,9496,5,m3.medium,4608,NaN,NaN,NaN,NaN,NaN,NaN,NaN
size_rcv,9496,5,m3.medium,4608,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tput_mean,9492.0,NaN,NaN,NaN,123.062324,0.0,54.964529,68.213818,162.621853,893.438737,102.842463
tput_std,9492.0,NaN,NaN,NaN,13.66043,0.0,4.684287,6.881577,17.279534,242.456998,19.578987
tput_min,9492.0,NaN,NaN,NaN,60.888525,0.0,1.430036,6.195764,89.608591,748.9375,100.845631
tput_max,9492.0,NaN,NaN,NaN,186.156266,0.0,63.43849,77.0957,172.492342,925.06324,198.3792


In [3]:
# ======================================================
# Build task datasets → materialize per-pair time splits
# Input: base_clean.parquet you already loaded as `df`
# Output:
#   prepared/throughput_supervised.parquet
#   prepared/latency_supervised.parquet
#   prepared/classification_supervised.parquet
#   splits/<task>/{train,val,test}.parquet
# ======================================================
from pathlib import Path
import pandas as pd
import numpy as np

# -------- CONFIG --------
MIN_RUNS_PER_PAIR = 30      # raise later if you like (e.g., 50)
NUM_LAGS = 3                # lag depth for throughput/latency
PAIR_KEYS = ["provider","region_snd_base","region_rcv_base"]
TIME_COL = "start"

# ensure dirs
Path("prepared").mkdir(parents=True, exist_ok=True)
Path("splits").mkdir(parents=True, exist_ok=True)

# -------- helpers --------
def _assert_keys(d: pd.DataFrame, keys):
    missing = [k for k in keys if k not in d.columns]
    if missing:
        raise ValueError(f"Missing required key columns: {missing}")

def _sort_for_time(d: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in (PAIR_KEYS + [TIME_COL]) if c in d.columns]
    if cols:
        return d.sort_values(cols).reset_index(drop=True)
    return d.reset_index(drop=True)

def _keep_dense_pairs(d: pd.DataFrame, min_runs=MIN_RUNS_PER_PAIR) -> pd.DataFrame:
    cnt = d.groupby(PAIR_KEYS).size()
    ok = set(cnt[cnt >= min_runs].index)
    idx_tuples = list(zip(*[d[k] for k in PAIR_KEYS]))
    mask = [t in ok for t in idx_tuples]
    return d.loc[mask].copy()

def _add_lags(d: pd.DataFrame, col: str, n: int) -> pd.DataFrame:
    d = d.copy()
    for L in range(1, n+1):
        d[f"{col}_lag{L}"] = d.groupby(PAIR_KEYS, sort=False)[col].shift(L)
    return d

def _split_time_ordered_per_pair(d: pd.DataFrame, train=0.6, val=0.2):
    # deterministic, per pair
    d = _sort_for_time(d)
    d = d.reset_index(drop=False)  # keep original row index
    tr, va, te = [], [], []
    for _, g in d.groupby(PAIR_KEYS, sort=False):
        n = len(g)
        if n < 10:  # too small → send all to train to avoid brittle splits
            tr += g["index"].tolist()
            continue
        n_tr = max(1, int(n*train))
        n_va = max(1, int(n*val))
        n_te = max(1, n - n_tr - n_va)
        idxs = g["index"].tolist()
        tr += idxs[:n_tr]
        va += idxs[n_tr:n_tr+n_va]
        te += idxs[n_tr+n_va:]
    return pd.Index(tr), pd.Index(va), pd.Index(te)

def _save_splits(d: pd.DataFrame, name: str):
    sp = Path(f"splits/{name}")
    sp.mkdir(parents=True, exist_ok=True)
    idx_tr, idx_va, idx_te = _split_time_ordered_per_pair(d)
    d.loc[idx_tr].to_parquet(sp / "train.parquet", index=False)
    d.loc[idx_va].to_parquet(sp / "val.parquet", index=False)
    d.loc[idx_te].to_parquet(sp / "test.parquet", index=False)
    print(f"[{name}] train/val/test = {len(idx_tr)}/{len(idx_va)}/{len(idx_te)}")

# -------- sanity: keys & sorting --------
_assert_keys(df, PAIR_KEYS + [TIME_COL])
df = _sort_for_time(df)

# ===== 1) Throughput (requires tput_mean) =====
if "tput_mean" in df.columns:
    thr = df.dropna(subset=["tput_mean"]).copy()
    thr = _add_lags(thr, "tput_mean", NUM_LAGS)
    thr["tput_mean_next"] = thr.groupby(PAIR_KEYS, sort=False)["tput_mean"].shift(-1)
    thr = thr.dropna(subset=["tput_mean_next"])
    thr = _keep_dense_pairs(thr).reset_index(drop=True)
    thr.to_parquet("prepared/throughput_supervised.parquet", index=False)
    print("✓ prepared/throughput_supervised.parquet", thr.shape)
    _save_splits(thr, "throughput")
else:
    print("✗ Skipped throughput: 'tput_mean' not found.")

# ===== 2) Latency (requires delay_mean) =====
if "delay_mean" in df.columns:
    lat = df.dropna(subset=["delay_mean"]).copy()
    lat = _add_lags(lat, "delay_mean", NUM_LAGS)
    lat["delay_mean_next"] = lat.groupby(PAIR_KEYS, sort=False)["delay_mean"].shift(-1)
    lat = lat.dropna(subset=["delay_mean_next"])
    lat = _keep_dense_pairs(lat).reset_index(drop=True)
    lat.to_parquet("prepared/latency_supervised.parquet", index=False)
    print("✓ prepared/latency_supervised.parquet", lat.shape)
    _save_splits(lat, "latency")
else:
    print("✗ Skipped latency: 'delay_mean' not found.")

# ===== 3) Classification (needs BOTH tput_mean & delay_mean) =====
if {"tput_mean","delay_mean"}.issubset(df.columns):
    clf = df.dropna(subset=["tput_mean","delay_mean"]).copy()
    # pair-wise medians for label
    med_t = clf.groupby(PAIR_KEYS)["tput_mean"].transform("median")
    med_d = clf.groupby(PAIR_KEYS)["delay_mean"].transform("median")
    clf["label_good"] = ((clf["tput_mean"] >= med_t) & (clf["delay_mean"] <= med_d)).astype("int8")
    # light history
    clf["tput_mean_lag1"]  = clf.groupby(PAIR_KEYS, sort=False)["tput_mean"].shift(1)
    clf["delay_mean_lag1"] = clf.groupby(PAIR_KEYS, sort=False)["delay_mean"].shift(1)
    clf = clf.dropna(subset=["tput_mean_lag1","delay_mean_lag1"])
    clf = _keep_dense_pairs(clf).reset_index(drop=True)
    clf.to_parquet("prepared/classification_supervised.parquet", index=False)
    print("✓ prepared/classification_supervised.parquet", clf.shape)
    _save_splits(clf, "classification")
else:
    print("✗ Skipped classification: need both 'tput_mean' and 'delay_mean'.")


/var/folders/_k/0wxcxm4956bgwrf2b1ch32rr0000gn/T/ipykernel_88278/2651493879.py:46: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  d[f"{col}_lag{L}"] = d.groupby(PAIR_KEYS, sort=False)[col].shift(L)
/var/folders/_k/0wxcxm4956bgwrf2b1ch32rr0000gn/T/ipykernel_88278/2651493879.py:46: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  d[f"{col}_lag{L}"] = d.groupby(PAIR_KEYS, sort=False)[col].shift(L)
/var/folders/_k/0wxcxm4956bgwrf2b1ch32rr0000gn/T/ipykernel_88278/2651493879.py:46: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False t

✓ prepared/throughput_supervised.parquet (8632, 35)
[throughput] train/val/test = 5175/1722/1735
✓ prepared/latency_supervised.parquet (8627, 35)
[latency] train/val/test = 5172/1720/1735
✓ prepared/classification_supervised.parquet (8627, 34)
[classification] train/val/test = 5172/1720/1735


/var/folders/_k/0wxcxm4956bgwrf2b1ch32rr0000gn/T/ipykernel_88278/2651493879.py:54: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for _, g in d.groupby(PAIR_KEYS, sort=False):


In [5]:
# ======================================================
# Anomaly Detection on summary-only base
# - Uses numeric summaries (tput_*, delay_mean, loss_*, duration)
# - Adds pair-relative deviations and simple deltas
# - Trains IsolationForest (unsupervised) and saves scores
# ======================================================
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

PAIR_KEYS = ["provider","region_snd_base","region_rcv_base"]
TIME_COL  = "start"

BASE_PATH = Path("/Users/nehaharish/github/Thesis-mlops/data/prepared/base_clean.parquet")
OUT_DATA  = "prepared/anomaly_matrix.parquet"
OUT_SCORES = "prepared/anomaly_scored.parquet"
OUT_TOP   = "prepared/anomaly_top10_per_pair.parquet"

Path("prepared").mkdir(parents=True, exist_ok=True)

# 1) Load base and keep only rows with required keys
base = pd.read_parquet(BASE_PATH).copy()
base = base.dropna(subset=[c for c in PAIR_KEYS if c in base.columns] + [TIME_COL]).reset_index(drop=True)

# 2) Select numeric candidates present in your schema
num_candidates = [
    # throughput summary
    "tput_mean","tput_std","tput_min","tput_max",
    "tput_pctl5","tput_pctl25","tput_median","tput_pctl75","tput_pctl95",
    # delay + loss summaries
    "delay_mean",
    "loss_mean","loss_std","loss_min","loss_max",
    "loss_pctl5","loss_pctl25","loss_median","loss_pctl75","loss_pctl95",
    # duration (sanity)
    "duration",
]
num_cols = [c for c in num_candidates if c in base.columns]

# 3) Pair-relative medians and deviations (robust to geo/provider differences)
def add_pair_relative(g: pd.DataFrame) -> pd.DataFrame:
    for c in num_cols:
        med = g[c].median()
        g[f"{c}_rel_med"] = g[c] - med          # deviation from pair median
    # simple time deltas on key metrics (catch sudden jumps)
    if "tput_mean" in g.columns:
        g["tput_mean_diff"] = g["tput_mean"].diff()
    if "delay_mean" in g.columns:
        g["delay_mean_diff"] = g["delay_mean"].diff()
    if "loss_mean" in g.columns:
        g["loss_mean_diff"] = g["loss_mean"].diff()
    return g

anom = base.sort_values(PAIR_KEYS + [TIME_COL]).groupby(PAIR_KEYS, observed=True, sort=False).apply(add_pair_relative)
anom = anom.reset_index(drop=True)

# 4) Build feature matrix (numeric only), fill small gaps
feat_cols = []
feat_cols += num_cols
feat_cols += [c for c in anom.columns if c.endswith("_rel_med")]
feat_cols += [c for c in ["tput_mean_diff","delay_mean_diff","loss_mean_diff"] if c in anom.columns]

X = anom[feat_cols].copy()
X = X.astype("float64")
X = X.fillna(X.median(numeric_only=True))   # robust small fill

# 5) Standardize (z-score) per feature — simple scaler
Xz = (X - X.mean(numeric_only=True)) / (X.std(numeric_only=True) + 1e-9)

# 6) Train IsolationForest (global) — unsupervised
iso = IsolationForest(
    n_estimators=400,
    contamination="auto",   # lets the model infer a sensible fraction
    random_state=42,
    n_jobs=-1,
)
iso.fit(Xz)
anom["anomaly_flag_iso"]  = (iso.predict(Xz) == -1).astype(int)
anom["anomaly_score_iso"] = iso.decision_function(Xz)  # lower = more anomalous

# 7) Optional: pair-relative z-score (simple, interpretable)
#    average absolute z across pair-relative features (not a model, just a score)
rel_cols = [c for c in feat_cols if c.endswith("_rel_med")]
if rel_cols:
    # per-feature z within each pair
    def pair_z(g):
        Z = (g[rel_cols] - g[rel_cols].mean(numeric_only=True)) / (g[rel_cols].std(numeric_only=True) + 1e-9)
        g["anomaly_score_pairZ"] = Z.abs().mean(axis=1)
        return g
    anom = anom.groupby(PAIR_KEYS, observed=True, sort=False).apply(pair_z).reset_index(drop=True)
else:
    anom["anomaly_score_pairZ"] = np.nan

# 8) Save the matrices
anom[PAIR_KEYS + [TIME_COL] + feat_cols].to_parquet(OUT_DATA, index=False)
anom.to_parquet(OUT_SCORES, index=False)
print("Saved:", OUT_DATA, anom.shape, "and", OUT_SCORES)

# 9) Produce a compact "top anomalies per pair" table (by both scorers)
def top_k_per_pair(df, score_col, k=5):
    return (
        df.sort_values([*PAIR_KEYS, score_col], ascending=[True, True, True, True, True, True, True, True, True, True, False])
          .groupby(PAIR_KEYS, observed=True, sort=False)
          .head(k)
          .reset_index(drop=True)
    )

top_iso   = top_k_per_pair(anom, "anomaly_score_iso",  k=5)
top_pairZ = top_k_per_pair(anom, "anomaly_score_pairZ", k=5)

# Merge a readable view (keep core metrics + scores)
view_cols = list(dict.fromkeys(
    PAIR_KEYS
    + [TIME_COL, "tput_mean","delay_mean","loss_mean","duration"]
    + ["anomaly_flag_iso","anomaly_score_iso","anomaly_score_pairZ"]
))
top = pd.concat([top_iso.assign(top_source="iso"),
                 top_pairZ.assign(top_source="pairZ")], ignore_index=True)

top = top[ [c for c in view_cols if c in top.columns] + ["top_source"] ] \
        .sort_values(PAIR_KEYS + ["top_source", TIME_COL])

top.to_parquet(OUT_TOP, index=False)
print("Saved:", OUT_TOP, top.shape)

# (Optional) quick peek
display(top.head(20))


FileNotFoundError: [Errno 2] No such file or directory: 'prepared'